In [1]:
import sys
sys.path.append("..")

In [2]:
import tqdm
import torch
import pickle
import warnings
import numpy as np
import pandas as pd
import torch.optim as optim
from copy import deepcopy
import plotly.express as px
import plotly.graph_objects as go

from src.data import *
from src.utils import *
from src.model import *
from src.recourse import *

warnings.filterwarnings('ignore')

In [3]:
def append_result(d, objective, loss, cost, m1_validity, wc_validity, m1_expectation, wc_expectation):
    d['cost'].append(cost)
    d['m1_validity'].append(m1_validity)
    d['wc_validity'].append(wc_validity)
    d['m1_probability'].append(m1_expectation)
    d['wc_probability'].append(wc_expectation)
    d['loss'].append(loss)
    d['J'].append(objective) 
    
def get_result(d, algorithm, seed, alpha, lamb, theta_0, theta_r):
    result = {
        'algorithm': algorithm, 
        'seed': seed,
        'alpha': alpha,
        'lambda': lamb,
        'theta_0': theta_0,
        'theta_r': theta_r
        }
    
    for key in d.keys():
        result[key] = np.mean(d[key])
    return result

In [4]:
def get_theta_adv_linf(X_0, X_r, theta_0, alpha, lamb):
    theta_adv = deepcopy(theta_0)
    for i in range(X_0.shape[1]):
        theta_r_min = deepcopy(theta_adv)
        theta_r_max = deepcopy(theta_adv)
        
        theta_r_min[i] -= alpha
        theta_r_max[i] += alpha
        weights_r_min, bias_r_min = theta_r_min[:-1], theta_r_min[[-1]]
        weights_r_max, bias_r_max = theta_r_max[:-1], theta_r_max[[-1]]
        J_min, J_max = [], []
        for xi in range(len(X_r)):
            x_0 = X_0[xi]
            x_r = X_r[xi]
            J = RecourseCost(x_0, lamb)
            j_min = J.eval(x_r, weights_r_min, bias_r_min)
            j_max = J.eval(x_r, weights_r_max, bias_r_max)
            J_min.append(j_min)
            J_max.append(j_max)
        
        
        if np.mean(J_min) > np.mean(J_max):
            theta_adv[i] -= alpha
        else:
            theta_adv[i] += alpha
    
    return theta_adv

In [5]:
def get_theta_adv_l1(X_0, X_r, theta_0, alpha, lamb):
    i_max = 0
    alpha_max = 0
    val_max = -np.inf

    for i in range(X_0.shape[1]):
        theta_r_min = deepcopy(theta_0)
        theta_r_max = deepcopy(theta_0)
        
        theta_r_min[i] -= alpha
        theta_r_max[i] += alpha
        weights_r_min, bias_r_min = theta_r_min[:-1], theta_r_min[[-1]]
        weights_r_max, bias_r_max = theta_r_max[:-1], theta_r_max[[-1]]
        J_min, J_max = [], []
        for xi in range(len(X_r)):
            x_0 = X_0[xi]
            x_r = X_r[xi]
            J = RecourseCost(x_0, lamb)
            j_min = J.eval(x_r, weights_r_min, bias_r_min)
            j_max = J.eval(x_r, weights_r_max, bias_r_max)
            J_min.append(j_min)
            J_max.append(j_max)
        
        if np.mean(J_min) > np.mean(J_max):
            if np.mean(J_min) > val_max:
                i_max = i
                alpha_max = -alpha
                val_max = np.mean(J_min).item()
        else:
            if np.mean(J_max) > val_max:
                i_max = i
                alpha_max = alpha
                val_max = np.mean(J_max).item()
    
    theta_adv = deepcopy(theta_0)
    theta_adv[i_max] = alpha_max
    return theta_adv

In [6]:
def evaluate_performance(X_0, X_r, theta_0, alpha, lamb, seed, algorithm, theta_adv_method='L-1'):
    results = {'cost': [], 'm1_validity': [], 'wc_validity': [], 'm1_probability': [], 'wc_probability': [], 'loss': [], 'J': []}
    
    weights_0, bias_0 = theta_0[:-1], theta_0[[-1]]
    if theta_adv_method=='L-1':
        theta_adv = get_theta_adv_l1(X_0, X_r, theta_0, alpha, lamb)
    else:
        theta_adv = get_theta_adv_linf(X_0, X_r, theta_0, alpha, lamb)
    
    weights_adv, bias_adv = theta_adv[:-1], theta_adv[[-1]]
        
    n = len(X_r)
    Y_0 = np.hstack((np.ones((n//2,)), np.zeros((n - n//2,))))
    
    clf = LR()
    clf.train(X_0, Y_0)
    clf.model.coef_ = weights_0.reshape(1,-1)
    clf.model.intercept_ = bias_0
    
    clf_adv = deepcopy(clf)
    clf_adv.model.coef_ = weights_adv.reshape(1,-1)
    clf_adv.model.intercept_ = bias_adv

    for i in tqdm.trange(n, desc=f'[{algorithm.capitalize()}] [ seed={seed} ] [ α={alpha} ] [ λ={lamb} ]', colour='#0091ff'):
        x_0 = X_0[i]
        x_r = X_r[i]
        J = RecourseCost(x_0, lamb)
        
        bce_loss, cost, price = J.eval(x_r, weights_adv, bias_adv, True)
        m1_validity = clf.predict(x_r.reshape(1,-1))[0]
        m1_probability = clf.predict_proba(x_r.reshape(1,-1))[0,1]
        
        wc_validity = clf_adv.predict(x_r.reshape(1,-1))[0]
        wc_probability = clf_adv.predict_proba(x_r.reshape(1,-1))[0,1]
        
        append_result(results, price, bce_loss, cost, m1_validity, wc_validity, m1_probability, wc_probability)
        
    return get_result(results, algorithm, seed, alpha, lamb, theta_0, theta_adv)

In [7]:
alphas = np.linspace(0,0.5,5)
lambdas = [0.1,0.3]


params = {}
# 'synthetic', 'german', 'sba'
params['data'] = 'synthetic'
# 'lr', 'nn'
params['base_model'] = 'lr'
params['seeds'] = range(5)
# TODO: add your method here, the method name should match the name in filepath
params['algorithms'] = ['Alg1', 'ROARLInf']

results = {
    'algorithm': [],
    'seed': [],
    'alpha': [],
    'lambda': [],
    'Cost': [],
    'Current Validity': [],
    'Worst Case Validity': [],
}

for algorithm in params['algorithms']:
    for seed in params['seeds']:
        for v_alpha in alphas:
            for v_lamb in lambdas:
                data = pd.read_pickle(f"../results/recourse/{params['base_model']}_{params['data']}_{algorithm}_{v_lamb}_{v_alpha}_{seed}.pkl")
                alpha = data["alpha"].unique().item()
                lamb = data["lambda"].unique().item()
                theta_0 = data["theta_0"].iloc[0]
                weights_0, bias_0, = theta_0[:-1], theta_0[[-1]]
                X_0 = np.stack(data["x_0"])
                X_r = np.stack(data["x_r"])
                res = evaluate_performance(X_0, X_r, theta_0, alpha, lamb, seed, algorithm, theta_adv_method="L-1" if "l1" in algorithm else "L-inf")
                results['algorithm'].append(algorithm)
                results['seed'].append(seed)
                results['Cost'].append(res['cost'])
                results['Current Validity'].append(res['m1_probability'])
                results['Worst Case Validity'].append(res['wc_probability'])
                results['alpha'].append(alpha)
                results['lambda'].append(lamb)

df_results = pd.DataFrame(results)

[Roarlinf] [ seed=4 ] [ α=0.5 ] [ λ=0.3 ]: 100%|██████████| 105/105 [00:00<00:00, 2514.60it/s]


In [8]:
df_results

,algorithm,seed,alpha,lambda,Cost,Current Validity,Worst Case Validity
0,Alg1,0,0.000,0.1,5.483979,0.949264,0.949264
1,Alg1,0,0.000,0.3,5.483979,0.949264,0.949264
2,Alg1,0,0.125,0.1,5.618646,0.960219,0.951897
3,Alg1,0,0.125,0.3,5.618646,0.960219,0.951897
4,Alg1,0,0.250,0.1,5.763446,0.969799,0.954160
...,...,...,...,...,...,...,...
95,ROARLInf,4,0.250,0.3,5.172935,0.884585,0.856453
96,ROARLInf,4,0.375,0.1,5.275841,0.903109,0.860509
97,ROARLInf,4,0.375,0.3,5.275841,0.903109,0.860509
98,ROARLInf,4,0.500,0.1,5.378237,0.919151,0.861667


In [15]:
df_graph = df_results.groupby(["alpha", "lambda", "algorithm"]).mean().reset_index()
df_graph

,alpha,lambda,algorithm,seed,Cost,Current Validity,Worst Case Validity
0,0.000,0.1,Alg1,2.0,5.476311,0.949277,0.949277
1,0.000,0.1,ROARLInf,2.0,4.881117,0.841360,0.841360
2,0.000,0.3,Alg1,2.0,5.476311,0.949277,0.949277
3,0.000,0.3,ROARLInf,2.0,4.881117,0.841360,0.841360
4,0.125,0.1,Alg1,2.0,5.641710,0.960159,0.951903
5,0.125,0.1,ROARLInf,2.0,4.976743,0.864492,0.850636
6,0.125,0.3,Alg1,2.0,5.641710,0.960159,0.951903
7,0.125,0.3,ROARLInf,2.0,4.976743,0.864492,0.850636
8,0.250,0.1,Alg1,2.0,5.785570,0.969700,0.954172
9,0.250,0.1,ROARLInf,2.0,5.070170,0.884037,0.855949


In [16]:
px.scatter(df_graph, x="Worst Case Validity", y="Cost", color="algorithm")